<a href="https://colab.research.google.com/github/abelunbound/fg_interactive_budget/blob/main/fg_budget_classification_production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Using DeBERTa, a Deep Learning model for detecting budget padding, in Nigeria's FG Budget Budget

In [ ]:
# Install nbformat if needed
pip install nbformat

# Clean the notebook
python -c "import nbformat; nb = nbformat.read('fg_budget_classification_production.ipynb', as_version=4); nb.metadata.pop('widgets', None); nbformat.write(nb, 'fg_budget_classification_production.ipynb')"

In [ ]:
# Install nbformat
!pip install nbformat

# Clean the notebook
import nbformat

# Read the notebook
with open('fg_budget_classification_production.ipynb', 'r') as f:
    nb = nbformat.read(f, as_version=4)

# Remove widget metadata
if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

# Save cleaned version
with open('fg_budget_classification_production.ipynb', 'w') as f:
    nbformat.write(nb, f)

print("✓ Notebook cleaned - widget metadata removed")

In [ ]:
# ============================================
# CELL 0a — IMPORTS
# ============================================

from sentence_transformers import CrossEncoder
import numpy as np
import pandas as pd

In [ ]:
# ============================================
# CELL 0b — LOAD MODEL
# ============================================

model = CrossEncoder('abelakeni/pfmtools-deberta-v3-fg-budget-v3-classifier')
print("Model loaded successfully")



In [ ]:
# ============================================
# CELL 1a — LOAD BUDGET FILE FOR ASSESSMENT
# ============================================
# Expected columns: code, project, mda_code, agency, amount


budget_df = pd.read_csv(
    'https://huggingface.co/datasets/abelakeni/fg-mda-objectives-2026-v1/resolve/main/correct_approved_2026_capital_budget.csv',
    # nrows=2000,
)

# Rename
# budget_df = budget_df.rename(columns={'mda_name_pdf': 'agency'})
# Drop rows with missing mandate or project text
budget_df = budget_df.dropna(subset=['ergp_line_item'])

print(f"Total projects loaded : {len(budget_df)}")
print(f"Agencies              : {budget_df['agency'].nunique()}")
print(f"Columns               : {list(budget_df.columns)}")

In [ ]:
len(budget_df)

In [ ]:
# # # =====================================================
# # # ONE TIME ONLY - APPEND TO HUGGING FACE
# # # =====================================================

from huggingface_hub import HfApi

from google.colab import userdata
token = userdata.get('HF_TOKEN')

# # api = HfApi()
# # api.upload_file(
# #     path_or_fileobj='fg_agencies_objectives_2026_jan.csv',
# #     path_in_repo='fg_agencies_objectives_2026_jan.csv',
# #     repo_id='abelakeni/fg-mda-objectives-2026-v1',
# #     repo_type='dataset',
# #     token=token
# # )
# print("Uploaded!")


# # Budget data
# api = HfApi()
# api.upload_file(
#     path_or_fileobj='correct_approved_2026_capital_budget.csv',
#     path_in_repo='correct_approved_2026_capital_budget.csv',
#     repo_id='abelakeni/fg-mda-objectives-2026-v1',
#     repo_type='dataset',
#     token=token
# )
# print("Uploaded!")



In [ ]:
# =====================================================
# CELL 1b — LOAD MDA OBJECTIVES MASTER FROM HUGGINGFACE
# =====================================================

# Master datasets has objectives of 876 federal MDAs only


import pandas as pd

mandate_df = pd.read_csv(
    'https://huggingface.co/datasets/abelakeni/fg-mda-objectives-2026-v1/resolve/main/fg_agencies_objectives_2026_jan.csv',
    # token='your-hf-token'  # only needed if repo is private
)




print(f"Mandate master loaded : {len(mandate_df)} agencies")
print(f"Columns               : {list(mandate_df.columns)}")
print("\n")
mandate_df.head(3)



In [ ]:
# ============================================
# CELL 2 — COVERAGE DIAGNOSTICS
# ============================================

budget_codes  = set(budget_df['mda_code'].unique())
mandate_codes = set(mandate_df['mda_code'].unique())

# MDA codes in budget with NO mandate in master
unmatched_codes = budget_codes - mandate_codes

# MDA codes in master with NO projects in budget
unused_mandate_codes = mandate_codes - budget_codes

# Budget rows with no mandate coverage
unmatched_budget_df = budget_df[budget_df['mda_code'].isin(unmatched_codes)].copy()

# Mandate rows with no budget occurrence
unused_mandate_df = mandate_df[mandate_df['mda_code'].isin(unused_mandate_codes)].copy()

print("=" * 50)
print("COVERAGE DIAGNOSTICS")
print("=" * 50)
print(f"Unique MDA codes in budget              : {len(budget_codes)}")
print(f"Unique MDA codes in mandate master      : {len(mandate_codes)}")
print(f"MDA in Budget with no MDA mandate entry : {len(unmatched_codes)}")
print(f"Projects in Budget with no mandate      : {len(unmatched_budget_df)}")
print(f"Agencies in master, no budget projects  : {len(unused_mandate_codes)}")
print(f"Agencies in master, no budget projects  : {len(unused_mandate_df)}")

print("=" * 50)

print(f"\nBudget projects excluded from classification:")
print(unmatched_budget_df[['code', 'ergp_line_item', 'mda_code', 'agency']].head(10))

print(f"\nMandate agencies with no budget occurrence:")
print(unused_mandate_df[['mda_code', 'mda_name']].head(10))

In [ ]:
unmatched_budget_df.tail()

In [ ]:
unused_mandate_df.head()

In [ ]:
# ===================================================
# CELL 3 — BUILD CLEAN BUDGET + MAP MANDATE COLUMNS
# ===================================================

# Budget rows that have a mandate in master — ready for classification
clean_budget_df = budget_df[budget_df['mda_code'].isin(mandate_codes)].copy()

# Build lookup maps from mandate master
mandate_map  = mandate_df.set_index('mda_code')['mandate_description'].to_dict()
mda_name_map = mandate_df.set_index('mda_code')['mda_name'].to_dict()

# Map mandate and mda_name into clean budget
clean_budget_df['mda_name'] = clean_budget_df['mda_code'].map(mda_name_map)
clean_budget_df['mandate']  = clean_budget_df['mda_code'].map(mandate_map)

print(f"Total budget projects               : {len(budget_df)}")
print(f"Projects with mandate (clean)       : {len(clean_budget_df)}")
print(f"Projects excluded (no mandate)      : {len(unmatched_budget_df)}")
print(f"\nClean budget columns: {list(clean_budget_df.columns)}")

clean_budget_df[['mda_code', 'mda_name', 'ergp_line_item', 'mandate']].head(3)

In [ ]:
# ============================================
# CELL 4 — PREPARE PAIRS
# ============================================

pairs = [
    (row['mandate'], row['ergp_line_item'])
    for idx, row in clean_budget_df.iterrows()
]

print(f"Total pairs prepared: {len(pairs)}")
print(f"\nSample pair:")
print(f"  Mandate : {pairs[3][0][:100]}...")
print(f"  Project : {pairs[3][1][:100]}...")

In [ ]:
# ============================================
# CELL 5 — BATCH INFERENCE
# ============================================

batch_size = 32
all_predictions = []

for i in range(0, len(pairs), batch_size):
    batch = pairs[i:i + batch_size]
    preds = model.predict(batch, show_progress_bar=False)
    all_predictions.extend(preds)

    # Progress indicator
    if (i // batch_size) % 10 == 0:
        print(f"Processed {min(i + batch_size, len(pairs))}/{len(pairs)} projects")

all_predictions = np.array(all_predictions)
print(f"\nInference complete. Output shape: {all_predictions.shape}")

In [ ]:
# ============================================
# CELL 6 — SOFTMAX + ASSIGN RESULTS
# ============================================

# Convert logits to probabilities
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / exp_x.sum(axis=1, keepdims=True)

probs = softmax(all_predictions)

# Add classification columns
clean_budget_df['predicted_class'] = np.argmax(all_predictions, axis=1)
clean_budget_df['prob_outside'] = probs[:, 0]   # class 0 = Outside Mandate
clean_budget_df['prob_within']  = probs[:, 1]   # class 1 = Within Mandate

# Human-readable label column
clean_budget_df['mandate_alignment'] = clean_budget_df['predicted_class'].map({
    0: 'Outside MDA Mandate',
    1: 'Within MDA Mandate'
})

# High confidence flag for priority review
clean_budget_df['flag_for_review'] = (
    (clean_budget_df['predicted_class'] == 0) &
    (clean_budget_df['prob_outside'] > 0.70)
)

print("Results assigned successfully")
print(clean_budget_df['mandate_alignment'].value_counts())
print(f"\nHigh-confidence violations flagged: {clean_budget_df['flag_for_review'].sum()}")

In [ ]:
# ============================================
# CELL 7 — REORDER COLUMNS + EXPORT
# ============================================

# Final column order matching your data structure
output_cols = [
    'code',
    'mda_code',
    'agency',
    'mda_name',
    'ergp_line_item',
    'status',
    'amount',
    'mandate_alignment',
    'predicted_class',
    'prob_outside',
    'prob_within',
    'flag_for_review'
]

# Only keep columns that exist (in case some are missing in your file)
output_cols = [c for c in output_cols if c in clean_budget_df.columns]

clean_budget_df_output = clean_budget_df[output_cols]

# Export
clean_budget_df_output.to_csv('correct_approved_capital_budget_2026_classified.csv', index=False)
print(f"Exported {len(clean_budget_df_output)} rows to correct_approved_capital_budget_2026_classified.csv")
print(f"\nColumn order in output:")
print(list(clean_budget_df_output.columns))

In [ ]:
# ============================================
# CELL 8 — SUMMARY STATS
# ============================================

total = len(clean_budget_df_output)
outside = (clean_budget_df_output['predicted_class'] == 0).sum()
within  = (clean_budget_df_output['predicted_class'] == 1).sum()
flagged = clean_budget_df_output['flag_for_review'].sum()

print("=" * 50)
print("CLASSIFICATION SUMMARY")
print("=" * 50)
print(f"Total projects classified : {total}")
print(f"Within MDA Mandate        : {within}  ({within/total:.1%})")
print(f"Outside MDA Mandate       : {outside} ({outside/total:.1%})")
print(f"Flagged for review (>70%) : {flagged} ({flagged/total:.1%})")
print("=" * 50)

# Per-agency breakdown
agency_summary = clean_budget_df_output.groupby('agency').agg(
    total_projects=('predicted_class', 'count'),
    outside_mandate=('predicted_class', lambda x: (x == 0).sum()),
    flagged=('flag_for_review', 'sum')
).sort_values('outside_mandate', ascending=False)

agency_summary.reset_index(inplace=True)
agency_summary.to_csv('budget_2026_agency_summary.csv', index=False)

print("\nTop 10 agencies by outside-mandate projects:")
# agency_summary.head(10)

In [ ]:
agency_summary.head(10)